In [23]:
import nbiatoolkit
print(nbiatoolkit.__version__)



0.29.2


In [24]:

import sys
import os
import numpy as np
import pandas as pd
sys.path.append(os.path.curdir + '/..')
from Dataset.custom_dataset import get_data_loaders
from pycox.evaluation import EvalSurv
from statsmodels.stats.proportion import proportion_confint
import Dataset
print(Dataset.__name__)



Dataset


In [25]:
#AttributeError: module 'numba' has no attribute 'core'

In [26]:
# def sample_from_dist(params, distributions):
#     """
#     Generates samples from specified distribution parameters.
    
#     Args:
#         params (torch.Tensor): Tensor of distribution parameters, expected shape [batch_size, 2]
#                                where params[:, 0] = alpha and params[:, 1] = lambda for Weibull.
#         distributions (list): List specifying distribution types, e.g., ['weibull'].

#     Returns:
#         numpy.ndarray: Array of generated samples from specified distributions.
#     """
#     samples = []
#     for dist in distributions:
#         if dist == 'weibull':
#             alpha, lam = params[:, 0], params[:, 1]  # Extract Weibull parameters
#             # Sample from Weibull distribution
#             weibull_samples = lam * (-torch.log(torch.rand_like(alpha))) ** (1 / alpha)
#             samples.append(weibull_samples.cpu().numpy())  # Convert samples to numpy
#         else:
#             raise NotImplementedError(f"Unknown distribution type: {dist}")
    
#     # Concatenate all samples along the batch dimension
#     return np.concatenate(samples, axis=0)

In [27]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder3D(nn.Module):
    def __init__(self, in_channels, base_channels):
        super(Encoder3D, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, base_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(base_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(base_channels, base_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=2)  # Downsample by 2
        )
    
    def forward(self, x):
        return self.encoder(x)
class Decoder3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Decoder3D, self).__init__()
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(in_channels, out_channels, kernel_size=2, stride=2),  # Upsample by 2
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.decoder(x)

class TimeDecoder(nn.Module):
    def __init__(self, input_dim):
        super(TimeDecoder, self).__init__()
        # Intermediate dimension
        intermediate_dim = input_dim * 2  # You can adjust this multiplier as necessary
        
        # Define a series of Linear layers and ReLUs
        self.layers = nn.Sequential(
            nn.Linear(input_dim, intermediate_dim),
            nn.ReLU(),
            nn.Linear(intermediate_dim  , intermediate_dim * 2 ),
            nn.ReLU(),
            nn.Linear(intermediate_dim * 2, intermediate_dim),
            nn.ReLU(),
            nn.Linear(intermediate_dim, intermediate_dim // 2),
            nn.ReLU(),
            nn.Linear(intermediate_dim // 2, 2)  # Outputs the shape and scale parameters
        )
        self.softplus = nn.Softplus() 

    def forward(self, z):
        return self.softplus(self.layers(z))
        #return self.layers(z)

# class TimeDecoder2(nn.Module):
#     def __init__(self, input_dim):
#         super(TimeDecoder2, self).__init__()
#         # Intermediate dimension
#         intermediate_dim = input_dim * 2  # You can adjust this multiplier as necessary
        
#         # Define a series of Linear layers and ReLUs
#         self.layers1 = nn.Sequential(
#             nn.Linear(input_dim, intermediate_dim),
#             nn.ReLU(),
#             nn.Linear(intermediate_dim, intermediate_dim // 2),
#             nn.ReLU(),
#             nn.Linear(intermediate_dim // 2, 1)  # Outputs the shape and scale parameters
#         )

#         self.layers2 = nn.Sequential(
#             nn.Linear(input_dim, intermediate_dim),
#             nn.ReLU(),
#             nn.Linear(intermediate_dim, intermediate_dim // 2),
#             nn.ReLU(),
#             nn.Linear(intermediate_dim // 2, 1)  # Outputs the shape and scale parameters
#         )

#         self.softplus = nn.Softplus() 

#     def forward(self, z):
#         alpha = self.softplus(self.layers1(z[:, 0]))
#         lam = self.softplus(self.layers2(z[:, 0]))
#         return torch.cat([alpha, lam]) 



In [28]:

class LatentSpaceGaussian(nn.Module):
    def __init__(self, latent_dim):
        super(LatentSpaceGaussian, self).__init__()
        self.latent_dim = latent_dim
        self.latent_params = 2 * latent_dim  # Two parameters are needed for each Gaussian distribution


    def get_latent_params(self, x):
        x = x.view(-1, 2, self.latent_dim)
        mu = x[:, 0, :]
        log_var = x[:, 1, :]
        return mu, log_var


    def sample_latent(self, latent_params):
        mu, log_var = latent_params
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)  # `randn_like` as we need the same size
        z = mu + (eps * std)  # sampling as if coming from the input space
        return z


    def kl_loss(self, latent_params):
        mu, log_var = latent_params
        kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())  # Kullback-Leibler divergence
        return kl / mu.shape[0]


In [29]:

class LogLikelihoodLossWithCensoring(nn.Module):
    def __init__(self, dist_type):
        """
        Initialize the log-likelihood loss with censoring class.
        
        Args:
            dist_type (str): Distribution type to use for survival prediction (currently only supports 'weibull').
        """
        super(LogLikelihoodLossWithCensoring, self).__init__()
        self.dist_type = dist_type

    # def forward(self, inputs, survival_times, risk_status):
    #     survival_times = torch.clamp(torch.squeeze(survival_times), min=1e-8)
    #     risk_status = torch.squeeze(risk_status)

    #     # Extract and clamp Weibull parameters
    #     alpha = torch.clamp(inputs[:, 0], min=1e-8, max=5)
    #     lam = torch.clamp(inputs[:, 1], min=1e-8)

    #     # Calculate stable survival log-probabilities
    #     safe_ratio = torch.clamp(survival_times / lam, min=1e-8, max=100)
    #     survival_term = -torch.pow(safe_ratio, alpha)
    #     hazard_term = torch.log(alpha / lam) + (alpha - 1) * torch.log(safe_ratio)

    #     # Debugging: Check for NaNs or Infs in intermediate calculations
    #     if torch.any(torch.isnan(survival_term)) or torch.any(torch.isinf(survival_term)):
    #         print("Warning: NaN or Inf detected in survival_term")
    #         print("alpha:", alpha)
    #         print("lam:", lam)
    #         print("survival_times:", survival_times)
    #         print("safe_ratio (survival_times / lam):", safe_ratio)

    #     if torch.any(torch.isnan(hazard_term)) or torch.any(torch.isinf(hazard_term)):
    #         print("Warning: NaN or Inf detected in hazard_term")
    #         print("alpha:", alpha)
    #         print("lam:", lam)
    #         print("survival_times:", survival_times)
    #         print("safe_ratio (survival_times / lam):", safe_ratio)

    #     # Apply censoring
    #     log_likelihood_loss = -torch.sum(hazard_term[risk_status > 0.5]) - torch.sum(survival_term)

    #     # Mean loss across batch
    #     loss = log_likelihood_loss / inputs.size(0)

    #     # Ensure finite loss
    #     if not torch.isfinite(loss):
    #         raise ValueError("Non-finite loss encountered, check inputs for stability issues.")

    #     return loss


    def forward(self, inputs, survival_times, risk_status):
        """
        Compute the log-likelihood loss with censoring.
        
        Args:
            inputs (Tensor): Weibull parameters (alpha and lambda) predicted by the model, shape [batch_size, 2].
            survival_times (Tensor): Actual survival times for each sample, shape [batch_size].
            risk_status (Tensor): Binary censoring indicators (1 if event occurred, 0 if censored), shape [batch_size].

        Returns:
            loss (Tensor): Mean log-likelihood loss with censoring.
        """
        survival_times = torch.squeeze(survival_times)
        risk_status = torch.squeeze(risk_status)

        # Extract Weibull parameters from inputs
        alpha = inputs[:, 0]
        lam = inputs[:, 1]

        # Calculate the survival log-probabilities
        survival_term = -torch.pow(survival_times / lam, alpha)
        #print(survival_term)
        # Calculate the hazard function where risk_status is 1
        hazard_term = torch.log(alpha / lam) + (alpha - 1) * torch.log(survival_times / lam)
        #print(hazard_term)
        # Apply the censoring: use hazard term only where risk status is 1 (event occurred)
        # Debugging: Check for NaNs or Infs in intermediate calculations
        if torch.any(torch.isnan(survival_term)) or torch.any(torch.isinf(survival_term)):
            print("Warning: NaN or Inf detected in survival_term")
            print("alpha:", alpha)
            print("lam:", lam)
            print("survival_times:", survival_times)
            print("survival_term:", survival_term)

        if torch.any(torch.isnan(hazard_term)) or torch.any(torch.isinf(hazard_term)):
            print("Warning: NaN or Inf detected in hazard_term")
            print("alpha:", alpha)
            print("lam:", lam)
            print("survival_times:", survival_times)
            print("hazard_term:", hazard_term)

        # Apply censoring
        log_likelihood_loss = -torch.sum(hazard_term[risk_status > 0.5]) - torch.sum(survival_term)

        # Mean loss across batch
        loss = log_likelihood_loss / inputs.size(0)

        # Ensure finite loss
        if not torch.isfinite(loss):
            raise ValueError("Non-finite loss encountered, check inputs for stability issues.")

        return loss


In [30]:
# # Create dummy variables
# batch_size = 5  # Example batch size
# inputs = torch.tensor([[2.0, 1.0],  # alpha, lambda for sample 1
#                        [1.5, 0.5],  # alpha, lambda for sample 2
#                        [3.0, 2.0],  # alpha, lambda for sample 3
#                        [1.0, 1.0],  # alpha, lambda for sample 4
#                        [2.5, 1.5]], # alpha, lambda for sample 5
#                       dtype=torch.float32)  # Weibull parameters (alpha, lambda)

# survival_times = torch.tensor([10.0, 5.0, 15.0, 20.0, 25.0], dtype=torch.float32)  # Survival times
# risk_status = torch.tensor([1, 1, 0, 1, 0], dtype=torch.float32)  # Risk status (1 if event occurred, 0 if censored)

# # Create an instance of the loss class
# loss_fn = LogLikelihoodLossWithCensoring(dist_type='weibull')

# # Compute the loss
# loss = loss_fn(inputs, survival_times, risk_status)

# # Print the loss
# print(f"Log-likelihood loss: {loss.item()}")

In [31]:
import torch
import torch.nn as nn

class UNet3D(nn.Module):
    def __init__(self, in_channels, out_channels, base_channels, latent_dim=None):
        super(UNet3D, self).__init__()
        
        self.latent_dim = latent_dim
        self.rec_loss = nn.MSELoss() 
        self.time_loss = LogLikelihoodLossWithCensoring(dist_type='weibull')
        
        # Encoder layers
        self.enc1 = Encoder3D(in_channels, base_channels)
        self.enc2 = Encoder3D(base_channels, base_channels * 2)
        self.enc3 = Encoder3D(base_channels * 2, base_channels * 4)
        self.enc4 = Encoder3D(base_channels * 4, base_channels * 8)
        
        if latent_dim is not None:
            # Separate layers for mu and logvar
            self.mu = nn.Linear(base_channels * 8 * 8 * 8 * 4, latent_dim)
            self.logvar = nn.Linear(base_channels * 8 * 8 * 8 * 4, latent_dim)
            self.fc_decode = nn.Linear(latent_dim, base_channels * 8 * 8 * 8 * 4)  # Reshape latent back to decoder input size
        
        # Decoder layers
        self.dec4 = Decoder3D(base_channels * 8, base_channels * 4)
        self.dec3 = Decoder3D(base_channels * 4, base_channels * 2)
        self.dec2 = Decoder3D(base_channels * 2, base_channels)
        self.dec1 = Decoder3D(base_channels, out_channels)

        if latent_dim is not None:
            self.time_decoder = TimeDecoder(latent_dim)
    
    def forward(self, x):
        batch_size, _, H, W, D = x.shape 
        out_data = {}  # Dictionary to store outputs
        
        # Encoding path
        enc1 = self.enc1(x)
        enc2 = self.enc2(enc1)
        enc3 = self.enc3(enc2)
        enc4 = self.enc4(enc3)
        
        if self.latent_dim is not None:
            # Flatten for mu and logvar calculations
            enc4_flat = enc4.view(enc4.size(0), -1)
            
            # Calculate `mu` and `log_var` using separate layers
            mu = self.mu(enc4_flat)
            log_var = self.logvar(enc4_flat)
            
            # Sample from latent space using reparameterization trick
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            z = mu + eps * std
            #print("z",z)
            # Decode the latent sample
            enc4_decoded = self.fc_decode(z).view(batch_size, -1, 8, 8, 4)
            print(enc4_decoded.shape)
            
            # Predict Weibull parameters
            weibull_params = self.time_decoder(z)
            print("weibull_params",weibull_params)
            # Calculate KL divergence
            kl_divergence = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
            
            # Add outputs to the dictionary
            out_data['latent_sample'] = z
            out_data['mu'] = mu
            out_data['log_var'] = log_var
            out_data['time_params'] = weibull_params
            out_data['kl_divergence'] = kl_divergence
            enc4 = enc4_decoded  # Use the decoded latent sample for further decoding
        
        # Decoding path
        dec4 = self.dec4(enc4)
        dec3 = self.dec3(dec4 + enc3)
        dec2 = self.dec2(dec3 + enc2)
        dec1 = self.dec1(dec2 + enc1)
        
        # Reconstruction output
        out_data['reconstruction'] = dec1
        
        return out_data


    @staticmethod
    def sample_from_dist(params, distributions):
        """
        Generates samples from specified distribution parameters.
        
        Args:
            params (torch.Tensor): Tensor of distribution parameters, expected shape [batch_size, 2]
                                where params[:, 0] = alpha and params[:, 1] = lambda for Weibull.
            distributions (list): List specifying distribution types, e.g., ['weibull'].

        Returns:
            numpy.ndarray: Array of generated samples from specified distributions.
        """
        samples = []
        for dist in distributions:
            if dist == 'weibull':
                alpha, lam = params[:, 0], params[:, 1]  # Extract Weibull parameters
                alpha = torch.clamp(alpha, min=0.2)  # Prevent alpha from being zero or negative
                lam = torch.clamp(lam, min=0.2)
                # Sample from Weibull distribution
                weibull_samples = lam * (-torch.log(torch.rand_like(alpha))) ** (1 / alpha)
                if (weibull_samples <= 0).any():
                    print("Non-positive samples detected:", weibull_samples[weibull_samples <= 0])

                samples.append(weibull_samples.detach().cpu().numpy())  # Convert samples to numpy
            else:
                raise NotImplementedError(f"Unknown distribution type: {dist}")
        
        # Concatenate all samples along the batch dimension
        return np.concatenate(samples, axis=0)

    def predict_time(self, x,device=torch.device('cuda')):
        """
        Predicts Weibull time distribution parameters and samples for survival analysis
        using only image data.
        
        Args:
            x (torch.Tensor): Input image tensor of shape [batch_size, channels, H, W, D].
            device (torch.device, optional): Device to run the computations on (e.g., 'cpu' or 'cuda').

        Returns:
            dict: Dictionary with keys 'z', 'time_params', and 'time_samples':
                - 'z': Latent representation from image input.
                - 'time_params': Predicted Weibull distribution parameters.
                - 'time_samples': Sampled time prediction based on Weibull distribution.
        """
        # Ensure input is on the correct device
        x = x.to(device)
        
        # Forward pass through the network to get latent representation and time parameters
        out_data = self.forward(x)
        
        # Get latent variable z from the output data dictionary
        z = out_data['latent_sample']  # Latent representation
        time_params = out_data['time_params']  # Weibull distribution parameters
        
        # Generate a survival time sample from the Weibull distribution
        time_samples = self.sample_from_dist(time_params, distributions=['weibull'])
        
        return {
            'z': z.detach().cpu().numpy(),  # Detach and convert to NumPy array
            'time_params': time_params.detach().cpu().numpy(),
            'time_samples': time_samples
        }


    def predict_just_time(self, x):
        """Predicts only the time samples based on the input covariates."""
        # Call the predict_time method to get time samples
        return self.predict_time(x)['time_samples']


    @staticmethod
    def bern_conf_interval(n, mean, ibs=False):
        # Confidence interval
        ci_bot, ci_top = proportion_confint(count=mean * n, nobs=n, alpha=0.1, method='beta')
        if mean < 0.5 and not ibs:
            ci_bot_2 = 1 - ci_top
            ci_top = 1 - ci_bot
            ci_bot = ci_bot_2
            mean = 1 - mean

        return np.round(ci_bot, 4), mean, np.round(ci_top, 4)

    @staticmethod
    def obtain_c_index(surv_f, time, censor):
        # Evaluate using PyCox c-index
        ev = EvalSurv(surv_f, time.flatten(), censor.flatten(), censor_surv='km')
        ci = ev.concordance_td()

        # Obtain also ibs
        time_grid = np.linspace(time.min(), time.max(), 100)
        ibs = ev.integrated_brier_score(time_grid)
        return ci, ibs
        
    def calculate_risk(self, time_train, x_val, censor_val):
        """Calculate risk CDF at specified time points, compute C-index and IBS with confidence intervals."""
        # time_train = time_train.detach().cpu().numpy()
        # times = np.unique(time_train)
        # Ensure time_train is a list of numerical values

        # Print the type of time_train
        print("Type of time_train:", type(time_train))
        
        # if isinstance(time_train, list):
        #     # Convert list to a numpy array for easier manipulation
        #     time_train_array = np.array(time_train, dtype=float)
        #     # Print the shape of the converted array
        #     print("Shape of time_train:", time_train_array.shape)
        #     times = np.unique(time_train_array)  # Use the array with np.unique
        # else:
        #     # If time_train is not a list, assume it's a NumPy array and print its shape
        #     print("Shape of time_train:", time_train.shape)
        #     times = np.unique(time_train) 


        time_train = time_train.detach().cpu().numpy()
        times = np.unique(time_train)
        out_data = self.predict_time(x_val)
        pred_risk = np.zeros((out_data['time_params'].shape[0], len(times)))

        for sample in range(pred_risk.shape[0]):
            # Directly assume the Weibull parameters are used
            alpha = out_data['time_params'][sample, 0]
            lam = out_data['time_params'][sample, 1]
            pred_risk[sample, :] = 1 - np.exp(-np.power(times / lam, alpha))

        # Calculate C-index and IBS along with confidence intervals
        ci, ibs = self.obtain_c_index(pd.DataFrame(1 - pred_risk.T, index=times), np.asarray(time_train), np.asarray(censor_val))
        ci_conf_intervals = self.bern_conf_interval(len(np.asarray(time_train)), ci)
        ibs_conf_intervals = self.bern_conf_interval(len(np.asarray(time_train)), ibs, ibs=True)

        return ci_conf_intervals, ibs_conf_intervals

    def fit_epoch(self, train_loader, test_loader, optimizer, device=None):
        # Automatically determine device if not provided
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        epoch_results = {
            'loss_tr': 0.0, 'loss_va': 0.0, 'kl_tr': 0.0, 'kl_va': 0.0,
            'll_img_tr': 0.0, 'll_img_va': 0.0, 'll_time_tr': 0.0, 'll_time_va': 0.0,
            'ci_va': 0.0, 'ibs_va': 0.0
        }

        # Train epoch
        self.train()
        for batch in train_loader:
            img_train_batch, time_train_batch, censor_train_batch = batch

            img_train_batch = img_train_batch.to(device)
            time_train_batch = time_train_batch.to(device)
            censor_train_batch = censor_train_batch.to(device)

            # Generate output params
            out = self.forward(img_train_batch)

            # Compute losses
            optimizer.zero_grad()
            #loss_kl = self.latent_space.kl_loss(out['latent_params'])
            loss_kl = out['kl_divergence']
            loss_img = 250 * self.rec_loss(out['reconstruction'], img_train_batch) / img_train_batch.size(0) # Assuming mask is not provided
            loss_time = self.time_loss(out['time_params'], time_train_batch, censor_train_batch)
            loss = loss_kl + loss_img + loss_time
            loss.backward()
            optimizer.step()

            # Save training data
            epoch_results['loss_tr'] += loss.item()
            epoch_results['kl_tr'] += loss_kl.item()
            epoch_results['ll_img_tr'] += loss_img.item()  # updated to use img
            epoch_results['ll_time_tr'] += loss_time.item()

        # Validation step
        self.eval()
        with torch.no_grad():
            for batch in test_loader:
                img_val, time_val, censor_val = batch

                img_val = img_val.to(device)
                time_val = time_val.to(device)
                censor_val = censor_val.to(device)

                out = self.forward(img_val)
                #loss_kl = self.latent_space.kl_loss(out['latent_params'])
                loss_kl = out['kl_divergence']
                loss_img = 250 * self.rec_loss(out['reconstruction'], img_val)/ img_val.size(0) # Assuming mask is not provided
                loss_time = self.time_loss(out['time_params'], time_val, censor_val)
                loss = loss_kl + loss_img + loss_time

                # Save validation data
                epoch_results['loss_va'] += loss.item()
                epoch_results['kl_va'] += loss_kl.item()
                epoch_results['ll_img_va'] += loss_img.item()  # updated to use img
                epoch_results['ll_time_va'] += loss_time.item()

        return epoch_results



    def fit(self, train_loader, test_loader, n_epochs= 100, lr=0.00005, batch_size=4, device=None):
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # Define optimizer
        optimizer = torch.optim.Adam(self.parameters(), lr=lr)

        # Track training stats
        training_stats = {
            'loss_tr': [], 'loss_va': [], 'kl_tr': [], 'kl_va': [], 
            'll_img_tr': [], 'll_img_va': [], 'll_time_tr': [], 'll_time_va': [],
            'ci_va': [], 'ibs_va': []
        }
        best_loss = np.inf
        best_model = None
        best_ci, best_ibs = 0.0, 1.0  # Initialize best values for metrics

        for epoch in range(n_epochs):
            print("epoch no", epoch)
            # Optionally adjust weights for KL and image reconstruction terms
            self.kl_w = 0.001  # Constant value for KL weight
            self.cov_w = 0.001  # Constant value for covariance weight

            # Training epoch
            epoch_results = self.fit_epoch(train_loader, test_loader, optimizer=optimizer, device=device)

            # Validation step
            self.eval()
            with torch.no_grad():
                total_loss_va, total_kl_va, total_img_va, total_time_va = 0, 0, 0, 0
                for img_val, survival_time_val, event_val in test_loader:
                    img_val = img_val.to(device)
                    survival_time_val = survival_time_val.to(device)
                    event_val = event_val.to(device)

                    out_val = self.forward(img_val)
                    loss_kl = out_val['kl_divergence']
                    loss_img = 250 * self.rec_loss(out_val['reconstruction'], img_val)/ img_val.size(0)
                    loss_time = self.time_loss(out_val['time_params'], survival_time_val, event_val)
                    loss_va = loss_kl + loss_img + loss_time

                    total_loss_va += loss_va.item()
                    total_kl_va += loss_kl.item()
                    total_img_va += loss_img.item()
                    total_time_va += loss_time.item()

                # Average validation losses
                n_val_batches = len(test_loader)
                epoch_results['loss_va'] = total_loss_va / n_val_batches
                epoch_results['kl_va'] = total_kl_va / n_val_batches
                epoch_results['ll_img_va'] = total_img_va / n_val_batches
                epoch_results['ll_time_va'] = total_time_va / n_val_batches

                # Calculate validation metrics
                survival_times = torch.cat([survival_time for _, survival_time, _ in test_loader], dim=0)
                images = torch.cat([img for img, _, _ in test_loader], dim=0)
                events = torch.cat([event for _, _, event in test_loader], dim=0)

                epoch_results['ci_va'], epoch_results['ibs_va'] = self.calculate_risk(survival_times, images, events)

                # Save stats
                for key in epoch_results:
                    training_stats[key].append(epoch_results[key])

                # Print progress at each epoch
                if epoch % 5 == 0:
                    print(f"Epoch {epoch} | Train Loss: {str(epoch_results['loss_tr'])} | "
                        f"Val Loss: {str(epoch_results['loss_va'])} | "
                        f"Val C-Index: {str(epoch_results['ci_va'])}")

                # Save the best model based on validation loss
                if epoch_results['loss_va'] < best_loss:
                    best_loss = epoch_results['loss_va']
                    best_model = self.state_dict()
                    best_ci = epoch_results['ci_va']
                    best_ibs = epoch_results['ibs_va']

        # Load the best model before returning and save the final model
        if best_model is not None:
            self.load_state_dict(best_model)
        
        # Append best metrics to stats
        training_stats['ci_va'].append(best_ci)
        training_stats['ibs_va'].append(best_ibs)

        # Save the final model to a file at the end of training
        torch.save({
            'model_state_dict': self.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss': best_loss,
            'best_ci': best_ci,
            'best_ibs': best_ibs,
            'training_stats': training_stats
        }, 'final_model.pth')

        return training_stats



In [32]:
import matplotlib.pyplot as plt
import numpy as np

def plot_training_validation_losses(training_stats, filename="training_validation_losses.png"):
    epochs = range(len(training_stats['loss_tr']))

    # Create a 2x2 grid of subplots for the training and validation losses
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))

    # 1st subplot: Training and Validation Loss
    axs[0, 0].plot(epochs, training_stats['loss_tr'], label='Train Loss', color='blue')
    axs[0, 0].plot(epochs, training_stats['loss_va'], label='Validation Loss', color='orange')
    axs[0, 0].set_title("Loss (Train vs Validation)")
    axs[0, 0].set_xlabel("Epochs")
    axs[0, 0].set_ylabel("Loss")
    axs[0, 0].legend()

    # 2nd subplot: KL Divergence Loss for Training and Validation
    axs[0, 1].plot(epochs, training_stats['kl_tr'], label='Train KL Loss', color='blue')
    axs[0, 1].plot(epochs, training_stats['kl_va'], label='Validation KL Loss', color='orange')
    axs[0, 1].set_title("KL Divergence Loss (Train vs Validation)")
    axs[0, 1].set_xlabel("Epochs")
    axs[0, 1].set_ylabel("KL Loss")
    axs[0, 1].legend()

    # 3rd subplot: Image Reconstruction Loss for Training and Validation
    axs[1, 0].plot(epochs, training_stats['ll_img_tr'], label='Train Image Loss', color='blue')
    axs[1, 0].plot(epochs, training_stats['ll_img_va'], label='Validation Image Loss', color='orange')
    axs[1, 0].set_title("Image Reconstruction Loss (Train vs Validation)")
    axs[1, 0].set_xlabel("Epochs")
    axs[1, 0].set_ylabel("Image Loss")
    axs[1, 0].legend()

    # 4th subplot: Time Loss for Training and Validation
    axs[1, 1].plot(epochs, training_stats['ll_time_tr'], label='Train Time Loss', color='blue')
    axs[1, 1].plot(epochs, training_stats['ll_time_va'], label='Validation Time Loss', color='orange')
    axs[1, 1].set_title("Time Loss (Train vs Validation)")
    axs[1, 1].set_xlabel("Epochs")
    axs[1, 1].set_ylabel("Time Loss")
    axs[1, 1].legend()

    # Adjust layout to avoid overlap and save the figure
    plt.tight_layout()
    plt.savefig(filename, dpi=300)  # Save figure with specified filename and dpi
    plt.show()

def plot_ci_ibs(training_stats, filename="ci_ibs_plot.png"):
    epochs = range(len(training_stats['ci_va']))

    # Extract the middle element (median) from each entry in ci_va and ibs_va
    ci_va_median = [np.sort(ci)[1] for ci in training_stats['ci_va']]
    ibs_va_median = [np.sort(ibs)[1] for ibs in training_stats['ibs_va']]

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, ci_va_median, label='Validation C-Index (Median)', color='green')
    plt.plot(epochs, ibs_va_median, label='Validation IBS (Median)', color='purple')
    plt.xlabel("Epochs")
    plt.ylabel("Metric Value")
    plt.title("Validation C-Index and IBS (Median Values)")
    plt.legend()

    # Save the figure with specified filename and dpi
    plt.savefig(filename, dpi=300)
    plt.show()

# Usage example:
# plot_training_validation_losses(training_stats, "training_validation_losses.png")
# plot_ci_ibs(training_stats, "ci_ibs_plot.png")


In [33]:
import torch

# Set device for training (GPU if available, otherwise CPU)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

# Initialize model and move it to the specified device
model = UNet3D(in_channels=1, out_channels=1, base_channels=16, latent_dim=128).to(device)

# Example usage
train_loader, test_loader = get_data_loaders(
    root_dir='/home/nikhil/g17p3/g17-p3/NBIA-Download',
    excel_path='/home/nikhil/g17p3/g17-p3/Dataset/crlm.xlsx',
    batch_size=16,
    test_size=0.25
)

# Now you can use train_loader and test_loader in your model
print("len(train_loader)", len(train_loader))
print("len(test_loader)", len(test_loader))

# Train the model using the `fit` method
training_stats = model.fit(train_loader, test_loader)



cuda:0


Unknown extension is not supported and will be removed


len(train_loader) 10
len(test_loader) 4
epoch no 0
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1198/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1004/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1154/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1057/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1087/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1098/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1031/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1075

is_monotonic is deprecated and will be removed in a future version. Use is_monotonic_increasing instead.


NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1153/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1084/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1050/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1134/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1051/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1098/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1074/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1023/nifti/output.nii.gz. Skipping conversion.
NIfTI fi

...

NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1150/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1039/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1184/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1176/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1188/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1169/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1072/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1080/nifti/output.nii.gz. Skipping conversion.
NIfTI fi

is_monotonic is deprecated and will be removed in a future version. Use is_monotonic_increasing instead.


NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1155/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1081/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1160/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1120/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1032/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1182/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1062/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1042/nifti/output.nii.gz. Skipping conversion.
NIfTI fi

is_monotonic is deprecated and will be removed in a future version. Use is_monotonic_increasing instead.


NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1027/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1013/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1112/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1017/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1189/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1142/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1134/nifti/output.nii.gz. Skipping conversion.
NIfTI file already exists at /home/nikhil/g17p3/g17-p3/NBIA-Download/CRLM-CT-1164/nifti/output.nii.gz. Skipping conversion.
NIfTI fi

In [ ]:
# batch size =16, lr =0.00001, e= 0.0, test_size=0.25
#currently 11:30: loss = loss_kl + (250 *loss_img)+ loss_time

In [ ]:
# Prepare the sliced training stats dictionary by removing the first two entries
sliced_training_stats = {key: values[2:] for key, values in training_stats.items()}

# Now call the plot functions with the sliced dictionary
plot_individual_losses(sliced_training_stats, filename="loss_plot_0.png")
plot_ci_ibs(sliced_training_stats, filename="metrics_plot_0.png")


In [ ]:
ep

In [ ]:
training_stats

{'loss_tr': [9240862426526.338,
  7362516687284.586,
  65541.85342786332,
  11247290.515487906,
  48016.166307087056,
  14353.040719113564,
  10053.355510956255,
  1648368.779185278,
  8745.012644980854,
  15649.134235901713,
  21787.545548191298,
  17948.47500636111,
  14241.932408487255,
  12385.0046616636,
  10815.063189419807,
  11810.587153454871,
  20043.12735558953,
  21634.181063573695,
  25422.791509801344,
  30193.522376394303,
  27925.23391811327,
  24118.390158451333,
  25463.998357486587,
  36050.49694072227,
  37804.44135012071,
  38800.97490190792,
  41117.11694151797,
  42549.96836010388,
  38184.484442857,
  37630.35959137416,
  35885.770729300384,
  33986.31760021789,
  32073.58810657572,
  30088.2372753848,
  32251.452225812238,
  55161.3967850714,
  64798.05862433591,
  68866.24204808094,
  64848.80765201188,
  60774.6243904775,
  56487.59056851553,
  52248.46133482404,
  49229.791807012356,
  57184.21223050244,
  65314.099521682765,
  61470.094869617395,
  56457.00

In [ ]:
Cell In[97], line 24
     23 def kl_loss(self, latent_params):
---> 24     mu, log_var = latent_params
     25     kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())  # Kullback-Leibler divergence
     26     return kl / mu.shape[0]
     
ValueError: not enough values to unpack (expected 2, got 1)

SyntaxError: invalid syntax (3921441514.py, line 1)

: 

In [ ]:
--> 242 loss_time = self.time_loss(out['time_params'], time_train_batch, censor_train_batch)
    243 loss = loss_kl + loss_img + loss_time
    244 loss.backward()

File ~/g17p3/g17-p3/g15p3/lib/python3.11/site-packages/torch/nn/modules/module.py:1518, in Module._wrapped_call_impl(self, *args, **kwargs)
   1516     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
...
     44 if not torch.isfinite(loss):
---> 45     raise ValueError("Non-finite loss encountered, check inputs for stability issues.")
     47 return loss

ValueError: Non-finite loss encountered, check inputs for stability issues.

In [ ]:
import torch

# Set parameters
in_channels = 1            # Number of input channels
out_channels = 1           # Number of output channels
base_channels = 16         # Base channels for the model
latent_dim = 8             # Latent dimension for the VAE

# Create an instance of the UNet3D model
model = UNet3D(in_channels, out_channels, base_channels, latent_dim)

# Create a random tensor with the shape [4, 1, 128, 128, 64]
input_tensor = torch.rand(4, 1, 128, 128, 64)

# Pass the random input through the model
output_data = model(input_tensor)



In [ ]:
# Print the output shapes
for key, value in output_data.items():
    
    print(f"{key} shape: {value.shape}")

In [ ]:
time_info = model.predict_time(input_tensor)

In [ ]:
print("Latent Vector (z):", time_info['z'])
print("Weibull Parameters (time_params):", time_info['time_params'])
print("Sampled Time (time_samples):", time_info['time_samples'])


In [ ]:
just_time_info = model.predict_just_time(input_tensor)
print("Sampled Time (time_samples):", time_info['time_samples'])

In [ ]:
import numpy as np
import torch

# Set a random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Define batch size
batch_size = 3

# Generate dummy time_train data (e.g., survival times)
# Let's assume time points are between 0 and 100
time_train = np.random.uniform(0, 100, size=batch_size)

# Generate dummy x_val data
# Shape: (3, 1, 128, 128, 64)
x_val = torch.rand(batch_size, 1, 128, 128, 64)

# Generate dummy censor_val data
# Let's assume a 50% censoring rate
#censor_val = np.random.randint(0, 2, size=batch_size)
censor_val = np.array([1, 1, 0]) 
# Print the shapes of the generated data
print("time_train shape:", time_train.shape)  # Expect (3,)
print("x_val shape:", x_val.shape)            # Expect (3, 1, 128, 128, 64)
print("censor_val shape:", censor_val.shape)  # Expect (3,)


In [ ]:
time_train

In [ ]:
censor_val

In [ ]:
risk = model.calculate_risk(time_train, x_val, censor_val)

In [ ]:
risk

In [ ]:
import scipy
print(scipy.__version__)


In [ ]:
from scipy.integrate import trapz

# Compute the integral using the trapezoidal rule
integral = trapz(y, x)
print(f"The integral using trapezoidal rule is approximately: {integral}")


In [ ]:
import scipy.integrate
print(dir(scipy.integrate))  # This will list all available functions and attributes


In [ ]:
from scipy.integrate import simps
import numpy as np
from scipy.integrate import simps

# Example data
x = np.linspace(0, 10, 100)
y = np.sin(x)

# Compute the integral using Simpson's rule
integral = simps(y, x)
print(f"The integral is approximately: {integral}")


In [ ]:
import torch
import torch.nn as nn
import numpy as np


class UNet3D(nn.Module):
    def __init__(self, in_channels, out_channels, base_channels, latent_dim=None):
        super(UNet3D, self).__init__()
        
        self.latent_dim = latent_dim
        
        # Encoder and decoder layers
        self.enc1 = Encoder3D(in_channels, base_channels)
        self.enc2 = Encoder3D(base_channels, base_channels * 2)
        self.enc3 = Encoder3D(base_channels * 2, base_channels * 4)
        self.enc4 = Encoder3D(base_channels * 4, base_channels * 8)
        
        if latent_dim is not None:
            self.latent_space = LatentSpaceGaussian(latent_dim)
            self.latent_mu = nn.Linear(base_channels * 8 * 8 * 8 * 4, self.latent_space.latent_params)
            self.fc_decode = nn.Linear(latent_dim, base_channels * 8 * 8 * 8 * 4)
        
        self.dec4 = Decoder3D(base_channels * 8, base_channels * 4)
        self.dec3 = Decoder3D(base_channels * 4, base_channels * 2)
        self.dec2 = Decoder3D(base_channels * 2, base_channels)
        self.dec1 = Decoder3D(base_channels, out_channels)

        if latent_dim is not None:
            self.time_decoder = TimeDecoder(latent_dim)
    
    def forward(self, x):
        enc1 = self.enc1(x)
        enc2 = self.enc2(enc1)
        enc3 = self.enc3(enc2)
        enc4 = self.enc4(enc3)
        
        if self.latent_dim is not None:
            enc4_flat = enc4.view(enc4.size(0), -1)
            latent_params = self.latent_mu(enc4_flat)
            mu, log_var = self.latent_space.get_latent_params(latent_params)
            z = self.latent_space.sample_latent((mu, log_var))
            enc4 = self.fc_decode(z)
            enc4 = enc4.view(enc4.size(0), -1, 8, 8, 4)
            kl_divergence = self.latent_space.kl_loss((mu, log_var))
            weibull_params = self.time_decoder(z)
        
        dec4 = self.dec4(enc4)
        dec3 = self.dec3(dec4 + enc3)
        dec2 = self.dec2(dec3 + enc2)
        dec1 = self.dec1(dec2 + enc1)

        if self.latent_dim is not None:
            return dec1, weibull_params, kl_divergence
        else:
            return dec1

    def predict_time(self, x, just_cov=False, device=torch.device('cpu')):
        """Predicts Weibull time distribution parameters and samples for survival analysis."""
        if not just_cov:
            cov = x.drop(['time', 'event'], axis=1)
        else:
            cov = x.copy()

        # Forward pass to get latent representations and time predictions
        out_params = self.forward(cov.to(device))
        z = out_params[1]  # Assuming 'z' is returned as the second output
        time_params = self.time_decoder(z)

        # Generate samples from time distribution
        time_samples = sample_from_dist(time_params, [self.time_dist])

        return {
            'z': z.detach().cpu().numpy(),
            'time_params': time_params.cpu().numpy(),
            'time_samples': time_samples
        }

    def predict_just_time(self, x):
        """Predicts only the time samples based on the input covariates."""
        return self.predict_time(x, just_cov=True)['time_samples']


    def calculate_risk(self, time_train, x_val, censor_val):
        """Calculate risk CDF at specified time points, compute C-index and IBS with confidence intervals."""
        times = np.unique(time_train)
        out_data = self.predict_time(x_val)
        pred_risk = np.zeros([out_data['time_params'].shape[0], len(times)])

        for sample in range(pred_risk.shape[0]):
            if self.time_dist[0] == 'weibull':
                alpha = out_data['time_params'][sample, 0]
                lam = out_data['time_params'][sample, 1]
                pred_risk[sample, :] = 1 - np.exp(-np.power(times / lam, alpha))
            else:
                raise RuntimeError('Unknown time distribution for risk calculation.')

        # Calculate C-index and IBS along with confidence intervals
        ci, ibs = obtain_c_index(pd.DataFrame(1 - pred_risk.T, index=times), np.array(x_val['time']), censor_val)
        ci_conf_intervals = bern_conf_interval(len(np.array(x_val['time'])), ci)
        ibs_conf_intervals = bern_conf_interval(len(np.array(x_val['time'])), ibs, ibs=True)

        return ci_conf_intervals, ibs_conf_intervals
     

    def fit_epoch(self, data, optimizer, batch_size=64, device=torch.device('cpu')):
        epoch_results = {'loss_tr': 0.0, 'loss_va': 0.0, 'kl_tr': 0.0, 'kl_va': 0.0, 'll_cov_tr': 0.0, 'll_cov_va': 0.0,
                         'll_time_tr': 0.0, 'll_time_va': 0.0, 'ci_va': 0.0, 'ibs_va': 0.0}

        for batch in range(n_batches):
    


    def fit(self, data, train_params):
        training_stats = {'loss_tr': [], 'loss_va': [], 'kl_tr': [], 'kl_va': [], 'll_cov_tr': [], 'll_cov_va': [],
                          'll_time_tr': [], 'll_time_va': [], 'ci_va': [], 'ibs_va': []}

        optimizer = torch.optim.Adam(self.parameters(), lr=train_params['lr'])

        for epoch in range(train_params['n_epochs']):       


In [ ]:
import torch

# Generate a random tensor
random_input = torch.tensor([100.0])
  # Scaling by 10 to get a wider range of values
softplus_output = torch.nn.functional.softplus(random_input)

random_input.item(), softplus_output.item()